# Replica scenes: our map against the ground truth

Shows one Replica scene's map next to Replica's ground truth: the scores, the map in Rerun with the GT objects' boxes
over ours, the graph from above with the GT boxes as dashed outlines, the video with SAM's tracks, our objects and the
GT objects side by side, and a text search.

1. **Set the scene** in the settings cell: which scene, SAM's prompts (`VOCAB`), the poses and the stride.
2. **Run all cells.** The pipeline step reuses what is already computed: with cached masks it only rebuilds the
   graph and scores it, in a minute or two. Prompts not run before mean SAM 3 and the rest of the stages, which take
   longer (the README's "Evaluating on Replica" has the commands for all 8 scenes at once).

Use the `EAI project` kernel (see the README's Setup). The pipeline is `run.py`; this notebook only calls it and shows
the result. `notebooks/phone.ipynb` does the same for phone video, which has no ground truth.

In [ ]:
import json
import subprocess
import sys

from data import footage, replica, vocab
from data.paths import ROOT_DIR

SCENE = "room_0"  # one of replica.SCENES
VOCAB = "indoor"  # SAM's prompts: "indoor" (data/vocab.py), a prompt file, or None for the scene's own GT classes
POSE_SOURCE = "vggt"  # "vggt": VGGT-Omega's depth and poses, from the video alone; "gt": Replica's
STRIDE = 20  # use every STRIDE-th of the 2000 frames

args = ["--source", f"replica/{SCENE}/00", "--stride", str(STRIDE), "--pose-source", POSE_SOURCE]
if VOCAB:
    args += ["--vocab", VOCAB]
run = footage.make_run(f"replica/{SCENE}/00", STRIDE, "sam3", POSE_SOURCE, vocab.tag(VOCAB))
objects = replica.load_objects(SCENE)
print(f"{SCENE}: {len(footage.frame_numbers(run))} frames, {len(objects)} GT objects")


def sh(*command: str) -> None:
    """Run a command in the repository and show its output here, without warnings and progress bars."""
    proc = subprocess.Popen(
        [sys.executable, *command], cwd=ROOT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    noise = ("Warning", "warnings.warn", "[transformers]", "Loading weights", "it/s]", "s/it]", "from_numpy")
    for line in proc.stdout:
        if line.strip() and not any(n in line for n in noise):
            print(line, end="")
    if proc.wait():
        raise RuntimeError(f"{command[0]} failed")

## Run the pipeline and score it

The scores compare SAM's raw tracks with our graph (the README's "Evaluating on Replica"): identities (IDF1, revisits), text
search with OpenLex3D's queries, and the nearest-object query.

In [ ]:
sh("run.py", *args)

## The map in Rerun, with the GT boxes

The point cloud fuses the depth of 40 frames, in metres with z up. Our objects are coloured boxes; Replica's GT
objects are the light grey ones (`world/gt` in the entity tree: hide it there, hover a box for its class). With
VGGT-Omega's poses the GT boxes are moved into the map's frame by the best fit of the camera path, so they sit a
little tilted. Scrub the `frame` timeline to move the camera along its path; the right view shows what it saw.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)  # torch.jit, via transformers

import rerun as rr

from views import rerun_map

rr.init("replica")
rr.notebook_show(height=750)
rerun_map.log_map(run)
rerun_map.log_gt_boxes(run)

## The graph from above

Our objects as their boxes seen from above, with their edges (arrows: "on", dashed grey: "near"), and the GT objects'
boxes as dashed black outlines. A coloured box far bigger than the outline it sits in is a box that grew into its
neighbours; an outline with no box is an object the map doesn't have.

In [ ]:
from views import top_down

top_down.plot(run, gt=True);

## The video: SAM's tracks, our objects and the GT objects

Left, SAM 3's tracks; middle, our graph's objects (a node takes the colour of its largest track); right, Replica's
objects. Walls, floors and ceilings are left out, as in the evaluation.

In [ ]:
from IPython.display import Video

sh("-m", "views.video", *args, "--panels", "tracks", "graph", "gt", "--width", "560")
Video(str(run.stage_dir("eval") / "tracks_graph_gt.mp4"), embed=True, width=960)

## Search the map by text

In [ ]:
sh("-m", "graph.query", *args, "striped pillow")

In [ ]:
sh("-m", "graph.query", *args, "cushion", "--near", "sofa")